In [1]:
"""
Generate all figures for Chapters 3 and 4 of the dissertation.
ALL numeric data below is transcribed verbatim from the executed notebook
AI_Surveillance_System_Notebook_3__1_.ipynb — nothing is invented.
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
import os

OUT = "figures"
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 200,
})

INK = "#1a1a1a"
BLUE = "#2E5A88"
TEAL = "#3E8E8C"
AMBER = "#C88A2E"
RED = "#B0483C"
GREY = "#8A8A8A"
LIGHT = "#D9E2EC"

# ----------------------------------------------------------------------
# REAL DATA FROM NOTEBOOK
# ----------------------------------------------------------------------
CLASSES = ["Intrusion", "Loitering", "Anomalous", "Normal"]

raw_counts = {"Intrusion": 8, "Loitering": 9, "Anomalous": 7, "Normal": 9}      # cell 6
excluded   = {"Intrusion": 1, "Loitering": 0, "Anomalous": 1, "Normal": 1}      # cell 13
train      = {"Intrusion": 5, "Loitering": 6, "Anomalous": 4, "Normal": 6}      # cell 14
val        = {"Intrusion": 1, "Loitering": 1, "Anomalous": 1, "Normal": 1}      # cell 14
test       = {"Intrusion": 1, "Loitering": 2, "Anomalous": 1, "Normal": 1}      # cell 14

train_loss = [1.4946, 1.1500, 0.9268, 0.6739, 0.4852, 0.5461, 1.3733, 0.7148,
              0.5214, 0.4723, 0.3644, 0.1725, 0.2536, 0.1441, 0.1733]           # cell 19
val_loss   = [1.4226, 1.3571, 1.2909, 1.1401, 1.7795, 1.3195, 1.5947, 1.5158,
              1.7098, 1.6236, 1.8262, 3.1549, 2.0562, 2.2503, 1.7044]           # cell 19
val_acc    = [0.50, 0.50, 0.50, 0.50, 0.25, 0.50, 0.25, 0.25,
              0.50, 0.50, 0.50, 0.25, 0.50, 0.25, 0.50]                         # cell 19

# rows = actual, cols = predicted  (cell 22)
cm = np.array([[0, 0, 1, 0],
               [1, 1, 0, 0],
               [0, 0, 1, 0],
               [0, 0, 0, 1]])

precision = [0.000, 1.000, 0.500, 1.000]                                        # cell 22
recall    = [0.000, 0.500, 1.000, 1.000]
f1        = [0.000, 0.667, 0.667, 1.000]

FPS = 105.30                                                                    # cell 27
LATENCY_MS = 9.50


def save(fig, name):
    path = os.path.join(OUT, name)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("wrote", path)


# ======================================================================
# CHAPTER 3 DIAGRAMS
# ======================================================================

def box(ax, x, y, w, h, text, fc, tc="white", fs=9, bold=True):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                                boxstyle="round,pad=0.02,rounding_size=0.08",
                                facecolor=fc, edgecolor="none"))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
            color=tc, fontsize=fs, fontweight="bold" if bold else "normal",
            linespacing=1.45)


def arrow(ax, x1, y1, x2, y2, color=GREY):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2),
                                 arrowstyle="-|>", mutation_scale=13,
                                 linewidth=1.5, color=color,
                                 shrinkA=0, shrinkB=0))


# --- Figure 3.1 : Five-phase research process ---------------------------
fig, ax = plt.subplots(figsize=(10.2, 2.6))
ax.set_xlim(0, 10); ax.set_ylim(0, 3); ax.axis("off")
phases = [
    ("Phase 1\nLiterature review\n& requirements", BLUE),
    ("Phase 2\nDataset acquisition\n& pre-processing", TEAL),
    ("Phase 3\nModel selection,\ntraining, optimisation", TEAL),
    ("Phase 4\nSystem development\n& integration", TEAL),
    ("Phase 5\nPerformance\nevaluation", AMBER),
]
w, h, gap = 1.72, 1.25, 0.28
for i, (label, c) in enumerate(phases):
    x = 0.15 + i * (w + gap)
    box(ax, x, 1.2, w, h, label, c, fs=7.4)
    if i < 4:
        arrow(ax, x + w, 1.82, x + w + gap, 1.82)
# feedback loop
ax.annotate("", xy=(1.0, 1.15), xytext=(8.6, 1.15),
            arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.3,
                            connectionstyle="arc3,rad=0.14", ls=(0, (4, 3))))
ax.text(4.9, 0.30, "Iterative refinement — evaluation results may prompt revision of earlier phases",
        ha="center", fontsize=7.6, color=RED, style="italic")
save(fig, "fig_3_1_research_process.png")


# --- Figure 3.2 : Three-layer system architecture -----------------------
fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.set_xlim(0, 10); ax.set_ylim(0, 12); ax.axis("off")

box(ax, 2.6, 10.55, 4.8, 1.0, "Video input\n(surveillance frame stream)", GREY, fs=8.6)
arrow(ax, 5.0, 10.55, 5.0, 9.95)

box(ax, 1.2, 8.35, 7.6, 1.6,
    "DETECTION LAYER\nYOLOv8 person detector (MS COCO weights)\nlocalises persons in each frame", BLUE, fs=8.6)
arrow(ax, 5.0, 8.35, 5.0, 7.75)

box(ax, 1.2, 5.55, 7.6, 2.2,
    "BEHAVIOUR-ANALYSIS LAYER\nResNet-50 spatial feature extractor (frozen)\n→ LSTM temporal model (hidden size 256)\n→ softmax over 4 behaviour classes", TEAL, fs=8.6)
arrow(ax, 5.0, 5.55, 5.0, 4.95)

box(ax, 1.2, 3.15, 7.6, 1.8,
    "DECISION LAYER\nRule-based evaluation of behaviour class\nagainst restricted-zone definitions", AMBER, fs=8.6)
arrow(ax, 5.0, 3.15, 5.0, 2.55)

box(ax, 2.0, 1.25, 6.0, 1.3,
    "Alert raised for HUMAN OPERATOR review\n(decision support — not autonomous action)", RED, fs=8.6)

save(fig, "fig_3_2_architecture.png")


# --- Figure 3.3 : Data pre-processing pipeline --------------------------
fig, ax = plt.subplots(figsize=(11.5, 2.9))
ax.set_xlim(0, 12); ax.set_ylim(0, 4); ax.axis("off")
steps = [
    ("Raw clip\n(.mp4/.mov)", GREY),
    ("Frame sampling\nat 5 fps", BLUE),
    ("YOLOv8\nperson detection", BLUE),
    ("Crop largest\nperson box", TEAL),
    ("Resize to\n224 × 224", TEAL),
    ("Normalise\n(ImageNet stats)", TEAL),
    ("Pad/truncate to\n16-frame sequence", AMBER),
]
w, h, gap = 1.44, 1.3, 0.20
for i, (label, c) in enumerate(steps):
    x = 0.15 + i * (w + gap)
    box(ax, x, 1.75, w, h, label, c, fs=6.9)
    if i < len(steps) - 1:
        arrow(ax, x + w, 2.40, x + w + gap, 2.40)
ax.text(6.0, 0.85, "Clips in which no person was detected in any sampled frame were excluded (3 of 33)",
        ha="center", fontsize=7.4, color=RED, style="italic")
save(fig, "fig_3_3_preprocessing.png")


# --- Figure 3.4 : CNN-LSTM behaviour classifier -------------------------
fig, ax = plt.subplots(figsize=(10.5, 3.4))
ax.set_xlim(0, 12); ax.set_ylim(0, 5); ax.axis("off")

# frame stack
for i in range(4):
    off = i * 0.16
    ax.add_patch(FancyBboxPatch((0.35 + off, 2.35 - off), 0.85, 1.0,
                                boxstyle="round,pad=0.01,rounding_size=0.05",
                                facecolor=LIGHT, edgecolor=BLUE, linewidth=0.9))
ax.text(1.05, 1.85, "16 cropped\nframes", ha="center", fontsize=7.2, color=INK)

arrow(ax, 2.05, 2.85, 2.55, 2.85)
box(ax, 2.55, 2.15, 2.05, 1.45, "ResNet-50\n(ImageNet, FROZEN)\nfc head removed", BLUE, fs=7.4)
ax.text(3.58, 1.80, "→ 2048-d feature\nper frame", ha="center", fontsize=6.8, color=GREY, style="italic")

arrow(ax, 4.60, 2.85, 5.10, 2.85)
box(ax, 5.10, 2.15, 2.05, 1.45, "LSTM\n1 layer\nhidden size 256", TEAL, fs=7.4)
ax.text(6.13, 1.80, "→ final timestep\noutput", ha="center", fontsize=6.8, color=GREY, style="italic")

arrow(ax, 7.15, 2.85, 7.65, 2.85)
box(ax, 7.65, 2.15, 1.85, 1.45, "Fully connected\n+ softmax", AMBER, fs=7.4)

arrow(ax, 9.50, 2.85, 10.00, 2.85)
labels = ["Intrusion", "Loitering", "Anomalous", "Normal"]
for i, l in enumerate(labels):
    ax.text(10.10, 3.42 - i * 0.40, "▸ " + l, fontsize=7.4, va="center", color=INK)

ax.text(6.0, 0.75, "Only the LSTM and classifier head are trained; the ResNet-50 backbone is frozen throughout",
        ha="center", fontsize=7.4, color=RED, style="italic")
save(fig, "fig_3_4_cnn_lstm.png")


# ======================================================================
# CHAPTER 4 FIGURES — REAL RESULTS
# ======================================================================

# --- Figure 4.1 : Dataset composition -----------------------------------
fig, ax = plt.subplots(figsize=(6.6, 3.6))
x = np.arange(len(CLASSES))
tr = [train[c] for c in CLASSES]
va = [val[c] for c in CLASSES]
te = [test[c] for c in CLASSES]
ex = [excluded[c] for c in CLASSES]

ax.bar(x, tr, 0.62, label="Training (21)", color=BLUE)
ax.bar(x, va, 0.62, bottom=tr, label="Validation (4)", color=TEAL)
ax.bar(x, te, 0.62, bottom=np.array(tr) + np.array(va), label="Test (5)", color=AMBER)
ax.bar(x, ex, 0.62, bottom=np.array(tr) + np.array(va) + np.array(te),
       label="Excluded — no person detected (3)", color="white", edgecolor=RED,
       linewidth=1.2, hatch="///")

for i in range(len(CLASSES)):
    tot = tr[i] + va[i] + te[i]
    ax.text(i, tot + ex[i] + 0.22, f"{tot} usable", ha="center", fontsize=8, color=INK, fontweight="bold")

ax.set_xticks(x); ax.set_xticklabels(CLASSES)
ax.set_ylabel("Number of clips")
ax.set_ylim(0, 13.6)
ax.legend(fontsize=7.6, frameon=False, loc="upper left")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)
save(fig, "fig_4_1_dataset_composition.png")


# --- Figure 4.2 : Training / validation loss ----------------------------
fig, ax = plt.subplots(figsize=(6.6, 3.8))
ep = np.arange(1, 16)
ax.plot(ep, train_loss, "-o", color=BLUE, ms=4, lw=1.8, label="Training loss")
ax.plot(ep, val_loss, "-s", color=RED, ms=4, lw=1.8, label="Validation loss")
ax.axvline(4, color=GREY, ls=":", lw=1.2)
ax.annotate("Validation loss minimum\n(epoch 4, 1.1401) — divergence begins",
            xy=(4, 1.1401), xytext=(4.9, 2.28), fontsize=7.4, color=INK,
            arrowprops=dict(arrowstyle="->", color=GREY, lw=1))
ax.annotate("Peak 3.1549\n(epoch 12)", xy=(12, 3.1549), xytext=(8.9, 2.68),
            fontsize=7.4, color=RED,
            arrowprops=dict(arrowstyle="->", color=RED, lw=1))
ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
ax.set_xticks(ep)
ax.legend(fontsize=8, frameon=False, loc="upper left")
ax.grid(alpha=0.25, linewidth=0.6); ax.set_axisbelow(True)
save(fig, "fig_4_2_loss_curves.png")


# --- Figure 4.3 : Validation accuracy -----------------------------------
fig, ax = plt.subplots(figsize=(6.6, 3.2))
ax.step(ep, val_acc, where="mid", color=TEAL, lw=2)
ax.plot(ep, val_acc, "o", color=TEAL, ms=5)
ax.axhline(0.25, color=RED, ls="--", lw=1.2)
ax.text(15.1, 0.25, "  chance\n  (4 classes)", fontsize=7.2, color=RED, va="center")
ax.set_ylim(0, 0.75); ax.set_xticks(ep)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation accuracy")
ax.grid(alpha=0.25, linewidth=0.6); ax.set_axisbelow(True)
save(fig, "fig_4_3_val_accuracy.png")


# --- Figure 4.4 : Confusion matrix --------------------------------------
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1.6)
for i in range(4):
    for j in range(4):
        v = cm[i, j]
        ax.text(j, i, str(v), ha="center", va="center", fontsize=13,
                fontweight="bold", color="white" if v > 0 else "#B7C4D1")
        if i != j and v > 0:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                       edgecolor=RED, linewidth=2.2))
ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES, fontsize=8.4)
ax.set_yticks(range(4)); ax.set_yticklabels(CLASSES, fontsize=8.4)
ax.set_xlabel("Predicted class"); ax.set_ylabel("Actual class")
for s in ax.spines.values():
    s.set_visible(False)
ax.text(0.5, -0.26, "Red outline = misclassification. Both errors are threat→threat;\n"
                    "no threat behaviour was classified as normal activity.",
        transform=ax.transAxes, ha="center", fontsize=7.4, color=RED, style="italic")
save(fig, "fig_4_4_confusion_matrix.png")


# --- Figure 4.5 : Per-class precision / recall / F1 ---------------------
fig, ax = plt.subplots(figsize=(6.8, 3.6))
x = np.arange(len(CLASSES)); w = 0.26
ax.bar(x - w, precision, w, label="Precision", color=BLUE)
ax.bar(x, recall, w, label="Recall", color=TEAL)
ax.bar(x + w, f1, w, label="F1-score", color=AMBER)
for i in range(4):
    for off, vals in [(-w, precision), (0, recall), (w, f1)]:
        ax.text(i + off, vals[i] + 0.025, f"{vals[i]:.2f}", ha="center", fontsize=6.8)
ax.axhline(0.25, color=RED, ls="--", lw=1.1)
ax.text(3.62, 0.27, "chance", fontsize=7, color=RED)
ax.set_xticks(x); ax.set_xticklabels(
    [f"{c}\n(n={s})" for c, s in zip(CLASSES, [1, 2, 1, 1])], fontsize=8.4)
ax.set_xlim(-0.55, 4.05)
ax.set_ylim(0, 1.16); ax.set_ylabel("Score")
ax.legend(fontsize=8, frameon=False, ncol=3, loc="upper left")
ax.grid(axis="y", alpha=0.25, linewidth=0.6); ax.set_axisbelow(True)
save(fig, "fig_4_5_per_class_metrics.png")


# --- Figure 4.6 : Real-time throughput comparison -----------------------
fig, ax = plt.subplots(figsize=(6.8, 3.7))
names = ["Practical real-time\nthreshold (Sec. 2.8)", "Malmir et al. (2021)\nYOLOv4, dedicated GPU",
         "Proposed system\nYOLOv8n, Apple M4 (MPS)"]
vals = [30, 45, FPS]
cols = [GREY, TEAL, BLUE]
bars = ax.barh(names, vals, color=cols, height=0.55)
for b, v in zip(bars, vals):
    ax.text(v + 2, b.get_y() + b.get_height() / 2, f"{v:g} FPS",
            va="center", fontsize=9, fontweight="bold", color=INK)
ax.set_xlim(0, 128)
ax.set_xlabel("Throughput (frames per second)")
ax.text(0.5, -0.30, f"Proposed-system figure is detection layer only "
                    f"({LATENCY_MS:.2f} ms mean latency per frame); end-to-end pipeline latency was not benchmarked.",
        transform=ax.transAxes, ha="center", fontsize=7.2, color=RED, style="italic")
ax.grid(axis="x", alpha=0.25, linewidth=0.6); ax.set_axisbelow(True)
ax.invert_yaxis()
save(fig, "fig_4_6_realtime_throughput.png")

print("\nAll figures written to", OUT)

wrote figures/fig_3_1_research_process.png
wrote figures/fig_3_2_architecture.png
wrote figures/fig_3_3_preprocessing.png
wrote figures/fig_3_4_cnn_lstm.png
wrote figures/fig_4_1_dataset_composition.png
wrote figures/fig_4_2_loss_curves.png
wrote figures/fig_4_3_val_accuracy.png
wrote figures/fig_4_4_confusion_matrix.png
wrote figures/fig_4_5_per_class_metrics.png
wrote figures/fig_4_6_realtime_throughput.png

All figures written to figures
